## Example of how to generate a synthetic tau-PET from the pre-trained model

In [ ]:
#imports
import pandas as pd
import joblib
from unet3d_v1 import build_tau_pet_unet
import tensorflow as tf
import sys
from pathlib import Path
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import ndimage
import nibabel as nib

SRC_DIR = Path().resolve().parents[1] / "src"
sys.path.append(str(SRC_DIR))
import utils.image_helpers as ih
import utils.plot_helpers as ph

In [ ]:
### Paths / config
# Specify the paths for loading MRI and tabular data for the test subject. Age and plasma p-tau217
# can be imputed to the training set mean values if missing

test_csv = "../../datasets/df_test.csv"
mri_root = "" #INSERT MRI root

# U-Net weights from trained model
weights_path = "weights_v1.hdf5"

fillna_age = 70.08492087077349 # mean age in training set
fillna_plasma = 0.34589147447495605 # mean plasma p-tau217 in training set

# scalers for z-scoring according to training set
scaler_age = joblib.load("scaler_age.joblib")
scaler_plasma = joblib.load("scaler_plasma.joblib")

In [ ]:
### Load and preprocess test dataframe
# Load data
df_test = pd.read_csv(test_csv)

# Impute missing values
df_test["age_imputed"] = df_test["age"].fillna(fillna_age)
df_test["plasma_ptau217_imputed"] = df_test["plasma_ptau217"].fillna(fillna_plasma)

# Transform values using training scalers
df_test["age_imputed"] = scaler_age.transform(df_test[["age_imputed"]])
df_test["plasma_ptau217_imputed"] = scaler_plasma.transform(df_test[["plasma_ptau217_imputed"]])

In [ ]:
# Select a single subject to test
case_idx = 9
case = df_test.iloc[case_idx]

#This should become name of MRI file path for that subject
subject_id = case["file_names"] 
mri_path = os.path.join(
    mri_root,
    subject_id,
    "jademarec.nii.gz"
)

print("Subject:", subject_id)
print("MRI path:", mri_path)
print("Age:", case["age"])
print("Plasma p-tau217:", case["plasma_ptau217"])

In [ ]:
# Load and preprocess MRI
mri_test = tf.py_function(
    func=ih.load_and_preprocess_nifti,
    inp=[mri_path],
    Tout=tf.float32
)

# Set expected shape without batch dimension
mri_test.set_shape([72, 90, 76, 1])

# Add batch dimension: [1, 72, 90, 76, 1]
mri_test = tf.expand_dims(mri_test, axis=0)

# Prepare tabular inputs
plasma_value = np.float32(case["plasma_ptau217_imputed"])
age_value = np.float32(case["age_imputed"])

plasma_test = tf.convert_to_tensor([plasma_value], dtype=tf.float32)
age_test = tf.convert_to_tensor([age_value], dtype=tf.float32)

In [ ]:
# Build model and load weights
model = build_tau_pet_unet(verbose=False)
model.load_weights(weights_path)

print("Model loaded successfully.")


In [ ]:
# Generate synthetic PET
prediction = model(
    [mri_test, plasma_test, age_test],
    training=False
)

# Remove batch and channel dimensions
image_pred = prediction[0, :, :, :, 0].numpy()

print("Synthetic image generated.")

In [ ]:
#plot synthetic PET
plt.rc('font', size=10)
figure, axs = plt.subplots(1,3,figsize=(8,3),sharey=False,sharex=False)
plt.subplots_adjust(wspace = 0.05)
cmap = ph.get_cmap_pet()
axs[0].imshow(ndimage.rotate(image_pred[:,45,:],90), vmin=0.5,vmax=3,cmap=cmap)
axs[1].imshow(ndimage.rotate(image_pred[31,:,:],90), vmin=0.5,vmax=3,cmap=cmap)
axs[2].imshow(ndimage.rotate(image_pred[:,:,30],90), vmin=0.5,vmax=3,cmap=cmap)
suvrs=plt.Normalize(0.5,2.8)
sm = plt.cm.ScalarMappable(cmap=cmap,norm=suvrs)
axs[1].figure.colorbar(sm,ax=axs,shrink=0.3,label='SUVR',aspect=15,pad=0.1,orientation='horizontal')
for i in range(3):
    axs[i].set_xticks([])
    axs[i].set_yticks([])

In [ ]:
# Save synthetic PET as NIfTI
output_dir = '' #specify output path

nifti_pred = nib.Nifti1Image(image_pred, affine=np.eye(4))

save_path = os.path.join(output_dir, f"{subject_id}_synthetic_tau_pet.nii.gz")
nib.save(nifti_pred, save_path)

print("Saved prediction to:", save_path)